# Generación de Texto Aplicada a Negocios
## Arquitecturas Generativas con T5 y GPT-2

**Objetivo:** Usar modelos generativos para automatizar tareas empresariales de creación de texto.

**Casos de negocio:**
1. Resumen automático de reportes e informes
2. Generación de descripciones de productos
3. Generación de respuestas para servicio al cliente
4. Comparación de outputs con diferentes configuraciones

---

## 1. Instalación de dependencias

In [2]:
# ============================================================
# INSTALACIÓN
# ============================================================

!pip install -q transformers torch datasets

print("Instalación completada")

Instalación completada


## 2. Carga de modelos: T5 y GPT-2

- **T5 (t5-small):** Modelo encoder-decoder. Toda tarea se formula como texto → texto.
- **GPT-2:** Modelo autoregresivo (decoder-only). Genera texto prediciendo la siguiente palabra.

---

In [7]:
# ============================================================
# CARGA DE MODELOS Y TOKENIZADORES
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,    # Para T5 (encoder-decoder)
    AutoModelForCausalLM,      # Para GPT-2 (decoder-only)
    pipeline                    # Interfaz simplificada
)

# --- T5 ---
print("Cargando T5...")
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
print(f"  T5 parámetros: {sum(p.numel() for p in t5_model.parameters()):,}")

# --- GPT-2 ---
print("Cargando GPT-2...")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")
print(f"  GPT-2 parámetros: {sum(p.numel() for p in gpt2_model.parameters()):,}")

# --- Pipelines (interfaz simplificada de Hugging Face) ---
summarizer = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct")
generator = pipeline("text-generation", model="gpt2")

print("\n Modelos cargados correctamente")

Cargando T5...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  T5 parámetros: 60,506,624
Cargando GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  GPT-2 parámetros: 124,439,808


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


 Modelos cargados correctamente


## 3. Caso de Negocio: Resumen Automático de Reportes

**Escenario:** El equipo directivo necesita leer docenas de reportes semanales.
T5 puede generar resúmenes ejecutivos automáticos.

---

In [8]:
# ============================================================
# CASO 1: RESUMEN DE REPORTES EMPRESARIALES CON T5
# ============================================================

# Reporte trimestral simulado
reporte_trimestral = """
The company reported total revenue of $4.2 billion for Q3 2024,
representing a 12% increase year-over-year. Operating expenses
grew by 8% to $2.8 billion, driven primarily by investments in
cloud infrastructure and AI research. Net income reached $890 million,
exceeding analyst expectations by 15%. The company also announced
a new partnership with Microsoft Azure to expand cloud services
in Latin America. Employee headcount increased by 2,000 to a total
of 45,000 globally. The board approved a $500 million share buyback
program for the next fiscal year.
"""

# Generar resumen con T5 usando pipeline
# max_length: longitud máxima del resumen
# min_length: longitud mínima del resumen
resumen = summarizer(
    reporte_trimestral,
    max_length=60,
    min_length=20,
    do_sample=False   # False = determinístico (greedy/beam search)
)

print("📊 REPORTE ORIGINAL:")
print(reporte_trimestral.strip())
print(f"\n📝 RESUMEN GENERADO POR T5:")
print(resumen[0]["summary_text"])
print(f"\n📏 Longitud original: {len(reporte_trimestral.split())} palabras")
print(f"📏 Longitud resumen: {len(resumen[0]['summary_text'].split())} palabras")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length', 'min_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📊 REPORTE ORIGINAL:
The company reported total revenue of $4.2 billion for Q3 2024,
representing a 12% increase year-over-year. Operating expenses
grew by 8% to $2.8 billion, driven primarily by investments in
cloud infrastructure and AI research. Net income reached $890 million,
exceeding analyst expectations by 15%. The company also announced
a new partnership with Microsoft Azure to expand cloud services
in Latin America. Employee headcount increased by 2,000 to a total
of 45,000 globally. The board approved a $500 million share buyback
program for the next fiscal year.

📝 RESUMEN GENERADO POR T5:


KeyError: 'summary_text'

In [ ]:
# ============================================================
# EFECTO DE LA LONGITUD DEL RESUMEN
# ============================================================
# Comparar resúmenes de diferente longitud

print("📊 COMPARACIÓN DE RESÚMENES CON DIFERENTE LONGITUD")
print("=" * 70)

for max_len in [20, 40, 60, 80]:
    resumen = summarizer(
        reporte_trimestral,
        max_length=max_len,
        min_length=5,
        do_sample=False
    )
    texto = resumen[0]["summary_text"]
    print(f"\nmax_length={max_len:2d} ({len(texto.split()):2d} words):")
    print(f"  {texto}")

## 4. Caso de Negocio 2: Generación de Descripciones de Productos

**Escenario:** El equipo de e-commerce necesita generar descripciones para
cientos de productos nuevos. GPT-2 puede generar borradores iniciales.

---

In [ ]:
# ============================================================
# CASO 2: GENERACIÓN DE DESCRIPCIONES DE PRODUCTOS CON GPT-2
# ============================================================

# Prompts que simulan el inicio de descripciones de productos
productos = [
    "Product: Premium wireless headphones. Description: These high-quality headphones feature",
    "Product: Organic Colombian coffee. Description: Sourced from the mountains of Huila, this premium coffee",
    "Product: Smart fitness watch. Description: Track your health goals with this advanced",
]

print("🛍️ DESCRIPCIONES DE PRODUCTOS GENERADAS")
print("=" * 70)

for prompt in productos:
    # Generar con temperatura media (balance entre creatividad y coherencia)
    resultado = generator(
        prompt,
        max_length=80,          # Longitud máxima en tokens
        temperature=0.7,        # 0.7 = balance creatividad/coherencia
        top_k=50,               # Considerar solo las 50 palabras más probables
        do_sample=True,         # Activar muestreo (no determinístico)
        num_return_sequences=1, # Generar 1 versión
        pad_token_id=gpt2_tokenizer.eos_token_id  # Evitar warning
    )

    texto_generado = resultado[0]["generated_text"]
    # Separar el prompt del texto generado
    generado = texto_generado[len(prompt):]

    print(f"\n📦 Prompt: {prompt}")
    print(f"   ✍️ Generado: {generado[:150]}...")

In [ ]:
# ============================================================
# EFECTO DE LA TEMPERATURA EN LA GENERACIÓN
# ============================================================
# Temperatura baja (0.2) = conservador/repetitivo
# Temperatura alta (1.2) = creativo/arriesgado

prompt = "Product: Smart home assistant. Description: This innovative device"

print("🌡️ EFECTO DE LA TEMPERATURA EN LA GENERACIÓN")
print("=" * 70)

for temp in [0.2, 0.5, 0.7, 1.0, 1.3]:
    resultado = generator(
        prompt,
        max_length=60,
        temperature=temp,
        do_sample=True,
        top_k=50,
        num_return_sequences=1,
        pad_token_id=gpt2_tokenizer.eos_token_id
    )
    generado = resultado[0]["generated_text"][len(prompt):]
    print(f"\nTemp={temp}: {generado[:120]}...")

## 5. Caso de Negocio 3: Generación de Respuestas para Servicio al Cliente

**Escenario:** Automatizar respuestas iniciales para consultas frecuentes de clientes.

---

In [ ]:
# ============================================================
# CASO 3: RESPUESTAS AUTOMÁTICAS DE SERVICIO AL CLIENTE CON T5
# ============================================================
# T5 puede manejar tareas de texto-a-texto con instrucciones

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("t5-small")
t5_mod = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Consultas de clientes simuladas
# Usamos T5 para generar resúmenes de las consultas (triaje automático)
consultas = [
    "I purchased a laptop from your store last week and the screen has a dead pixel in the center. I would like to request a replacement or refund. My order number is ORD-2024-5847.",
    "I have been trying to reach your support team for three days regarding my internet service outage in downtown area. The service has been intermittent since Monday.",
    "Your new mobile banking app keeps crashing whenever I try to make a transfer. I am running Android 14 on a Samsung Galaxy S24. Please fix this urgent issue.",
]

print("🎧 TRIAJE AUTOMÁTICO DE CONSULTAS DE CLIENTES")
print("=" * 70)

for i, consulta in enumerate(consultas):
    # Generar resumen de la consulta para clasificación rápida
    input_text = "summarize: " + consulta
    inputs = t5_tok(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = t5_mod.generate(**inputs, max_length=40, num_beams=4)
    resumen = t5_tok.decode(outputs[0], skip_special_tokens=True)

    print(f"\n📩 Consulta {i+1}: {consulta[:80]}...")
    print(f"   📋 Resumen automático: {resumen}")

## 6. Comparación de parámetros: Top-k, Top-p y Beam Search

Experimentamos con diferentes estrategias de decodificación y su impacto
en la calidad del texto generado.

---

In [ ]:
# ============================================================
# COMPARACIÓN DE ESTRATEGIAS DE DECODIFICACIÓN
# ============================================================

prompt = "The quarterly business report shows that"

estrategias = {
    "Greedy (sin muestreo)": {
        "do_sample": False,
        "num_beams": 1,
    },
    "Beam Search (5 beams)": {
        "do_sample": False,
        "num_beams": 5,
        "no_repeat_ngram_size": 2,
    },
    "Top-k = 10 (restrictivo)": {
        "do_sample": True,
        "top_k": 10,
        "temperature": 0.7,
    },
    "Top-k = 50 (flexible)": {
        "do_sample": True,
        "top_k": 50,
        "temperature": 0.7,
    },
    "Nucleus (top-p = 0.9)": {
        "do_sample": True,
        "top_p": 0.9,
        "temperature": 0.7,
    },
    "Top-k + Top-p combinado": {
        "do_sample": True,
        "top_k": 50,
        "top_p": 0.9,
        "temperature": 0.8,
    },
}

print("⚙️ COMPARACIÓN DE ESTRATEGIAS DE DECODIFICACIÓN")
print("=" * 70)
print(f"Prompt: \"{prompt}\"\n")

for nombre, params in estrategias.items():
    resultado = generator(
        prompt,
        max_length=50,
        num_return_sequences=1,
        pad_token_id=gpt2_tokenizer.eos_token_id,
        **params
    )
    generado = resultado[0]["generated_text"][len(prompt):]
    print(f"📌 {nombre}:")
    print(f"   {generado[:100]}")
    print()

## 7. Pipeline completo: Generar + Evaluar (preparación para Sesión 3)

Combinamos generación con evaluación preliminar.

---

In [ ]:
# ============================================================
# PIPELINE COMPLETO: GENERACIÓN + EVALUACIÓN PRELIMINAR
# ============================================================

reporte = """
Amazon reported record-breaking revenue of $143.1 billion in Q3 2024.
Cloud computing division AWS grew 12 percent to $23.1 billion.
The company hired 50000 new employees and opened 3 new fulfillment
centers in Texas, Ohio and Georgia. CEO Andy Jassy announced plans
to invest $4 billion in AI infrastructure over the next two years.
"""

referencia = "Amazon earned $143.1B in Q3 2024. AWS grew 12%. 50K new hires. $4B AI investment planned."

# Generar resumen con T5
resumen_t5 = summarizer(reporte, max_length=50, min_length=15, do_sample=False)
texto_t5 = resumen_t5[0]["summary_text"]

# Generar continuación con GPT-2
resultado_gpt2 = generator(
    reporte.strip()[:100],
    max_length=150,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    pad_token_id=gpt2_tokenizer.eos_token_id
)
texto_gpt2 = resultado_gpt2[0]["generated_text"][:200]

print("📊 COMPARACIÓN T5 vs GPT-2")
print("=" * 70)
print(f"\n📄 ORIGINAL ({len(reporte.split())} palabras):")
print(f"   {reporte.strip()[:150]}...")
print(f"\n📝 T5 RESUMEN ({len(texto_t5.split())} palabras):")
print(f"   {texto_t5}")
print(f"\n🤖 GPT-2 CONTINUACIÓN ({len(texto_gpt2.split())} palabras):")
print(f"   {texto_gpt2[:200]}...")
print(f"\n🎯 REFERENCIA HUMANA:")
print(f"   {referencia}")
print("\n⏩ En la Sesión 3 evaluaremos estas salidas con BLEU, ROUGE y BERTScore")

## 8. Resumen y conclusiones de la Sesión 2

### Lo que aprendimos:
- **T5** es ideal para tareas con formato definido (resumen, traducción, triaje)
- **GPT-2** es mejor para generación libre y creativa (descripciones, continuaciones)
- **Temperatura** controla la creatividad vs conservadurismo
- **Top-k y Top-p** limitan el vocabulario de selección
- **Beam Search** busca la secuencia más probable (determinístico)
- **pipeline()** de Hugging Face simplifica enormemente el uso de modelos

### Aplicaciones de negocio cubiertas:
1. ✅ Resumen automático de reportes ejecutivos
2. ✅ Generación de descripciones de productos
3. ✅ Triaje automático de consultas de servicio al cliente
4. ✅ Comparación de estrategias de decodificación
